In [1]:
import pandas as pd
import numpy as np

base = r"C:\Users\rushikesh\Downloads\ecommerce-churn\Data"

orders    = pd.read_csv(base + r"\olist_orders_dataset1.csv")
customers = pd.read_csv(base + r"\olist_customers_dataset.csv")
items     = pd.read_csv(base + r"\olist_order_items_dataset.csv")
payments  = pd.read_csv(base + r"\olist_order_payments_dataset.csv")
reviews   = pd.read_csv(base + r"\olist_order_reviews_dataset.csv")

# Convert dates
date_cols = ['order_purchase_timestamp', 'order_approved_at',
             'order_delivered_carrier_date', 'order_delivered_customer_date',
             'order_estimated_delivery_date']
for col in date_cols:
    orders[col] = pd.to_datetime(orders[col])

print("Loaded ✅")

Loaded ✅


In [2]:
# We only care about completed orders for churn analysis
orders_clean = orders[orders['order_status'] == 'delivered'].copy()

print("Before:", orders.shape[0], "orders")
print("After (delivered only):", orders_clean.shape[0], "orders")
print("Dropped:", orders.shape[0] - orders_clean.shape[0], "non-delivered orders")

Before: 99441 orders
After (delivered only): 96478 orders
Dropped: 2963 non-delivered orders


In [3]:
# Step by step merging
df = orders_clean.merge(customers, on='customer_id', how='left')
df = df.merge(payments, on='order_id', how='left')
df = df.merge(items, on='order_id', how='left')
df = df.merge(reviews[['order_id','review_score']], on='order_id', how='left')

print("Master dataframe shape:", df.shape)
print("\nColumns:", df.columns.tolist())

Master dataframe shape: (115723, 23)

Columns: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value', 'review_score']


In [4]:
# Each customer should appear only ONCE with their summary stats
customer_df = df.groupby('customer_unique_id').agg(
    total_orders    = ('order_id', 'nunique'),
    total_spent     = ('payment_value', 'sum'),
    avg_spent       = ('payment_value', 'mean'),
    total_items     = ('product_id', 'count'),
    avg_review      = ('review_score', 'mean'),
    first_purchase  = ('order_purchase_timestamp', 'min'),
    last_purchase   = ('order_purchase_timestamp', 'max')
).reset_index()

print("Customers in dataset:", customer_df.shape[0])
print(customer_df.head())

Customers in dataset: 93358
                 customer_unique_id  total_orders  total_spent  avg_spent  \
0  0000366f3b9a7992bf8c76cfdf3221e2             1       141.90     141.90   
1  0000b849f77a49e4a4ce2b2a4ca5be3f             1        27.19      27.19   
2  0000f46a3911fa3c0805444483337064             1        86.22      86.22   
3  0000f6ccb0745a6a4b88665a16c9f078             1        43.62      43.62   
4  0004aac84e0df4da2b147fca70cf8255             1       196.89     196.89   

   total_items  avg_review      first_purchase       last_purchase  
0            1         5.0 2018-05-10 10:56:27 2018-05-10 10:56:27  
1            1         4.0 2018-05-07 11:11:27 2018-05-07 11:11:27  
2            1         3.0 2017-03-10 21:05:03 2017-03-10 21:05:03  
3            1         4.0 2017-10-12 20:29:41 2017-10-12 20:29:41  
4            1         5.0 2017-11-14 19:45:42 2017-11-14 19:45:42  


In [5]:
# Define churn: customer who never came back after their first purchase
# If total_orders == 1 → churned
# If total_orders > 1  → retained

customer_df['churned'] = (customer_df['total_orders'] == 1).astype(int)

print("Churned customers:  ", customer_df['churned'].sum())
print("Retained customers: ", (customer_df['churned'] == 0).sum())
print("Churn rate:         ", 
      round(customer_df['churned'].mean() * 100, 2), "%")

Churned customers:   90557
Retained customers:  2801
Churn rate:          97.0 %


In [6]:
# Reference date = day after last purchase in dataset
reference_date = customer_df['last_purchase'].max() + pd.Timedelta(days=1)

customer_df['recency_days'] = (reference_date - customer_df['last_purchase']).dt.days
customer_df['tenure_days']  = (customer_df['last_purchase'] - customer_df['first_purchase']).dt.days

print("Features added ✅")
print(customer_df[['total_orders','total_spent','recency_days','tenure_days','churned']].describe())

Features added ✅
       total_orders    total_spent  recency_days   tenure_days       churned
count  93358.000000   93358.000000  93358.000000  93358.000000  93358.000000
mean       1.033420     212.964557    237.941773      2.634032      0.969997
std        0.209097     646.223866    152.591453     24.955822      0.170596
min        1.000000       0.000000      1.000000      0.000000      0.000000
25%        1.000000      63.830000    114.000000      0.000000      1.000000
50%        1.000000     113.140000    219.000000      0.000000      1.000000
75%        1.000000     202.637500    346.000000      0.000000      1.000000
max       15.000000  109312.640000    714.000000    633.000000      1.000000


In [7]:
customer_df.to_csv(base + r"\customer_master.csv", index=False)
print("Saved to customer_master.csv ✅")
print("Final shape:", customer_df.shape)
print("\nSample:")
print(customer_df.head(3))

Saved to customer_master.csv ✅
Final shape: (93358, 11)

Sample:
                 customer_unique_id  total_orders  total_spent  avg_spent  \
0  0000366f3b9a7992bf8c76cfdf3221e2             1       141.90     141.90   
1  0000b849f77a49e4a4ce2b2a4ca5be3f             1        27.19      27.19   
2  0000f46a3911fa3c0805444483337064             1        86.22      86.22   

   total_items  avg_review      first_purchase       last_purchase  churned  \
0            1         5.0 2018-05-10 10:56:27 2018-05-10 10:56:27        1   
1            1         4.0 2018-05-07 11:11:27 2018-05-07 11:11:27        1   
2            1         3.0 2017-03-10 21:05:03 2017-03-10 21:05:03        1   

   recency_days  tenure_days  
0           112            0  
1           115            0  
2           537            0  
